###### Imports and Settings

In [2]:
import pandas as pd
import geopandas as gpd
import numpy as np
import requests
import io
import pickle
import matplotlib.pyplot as plt
from collections import deque
%matplotlib inline
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 150)

###### Functions

In [3]:
#function for percent of whole
def percent(x, y):
    try:
        return round((x/y)*100, 2)
    except ZeroDivisionError:
        return 666
#function for population density
def populationdensity(x, y):
    try:
        return round(x/y, 2)
    except ZeroDivisionError:
        return 666

# Comprehensive Plan Data Pull  

The following API calls are designed to streamline the data pulls for the comprehensive plans for any geography. For the sake of simplicity, there are three separate documents for ACS variables, Decennial 2000 SF3 variables, and Decennial SF1 variables. This document contains ACS 5-Year Estimate variables.

In [4]:
#to read in... rb is read bite
with open('api_keys.pkl', 'rb') as keys_file:
        keys_dict_2 = pickle.load(keys_file)

In [5]:
#create a variable that contains your api key
api_key = keys_dict_2['CENSUS']

In [6]:
GNRC = ['161', #Stewart
       '125', #Montgomery
       '083', #Houston
       '085', #Humphreys
       '043', #Dickson
       '021', #Cheatham
       '147', #Robertson
       '165', #Sumner
       '037', #Davidson
       '189', #Wilson
       '169', #Trousdale
       '187', #Williamson
       '149'] #Rutherford

## Read In Data Guide

The "head" should never be more than 2 variables, and the "tail" never more than 2. Since we can pull 50 variables at once this means that we can systematically program in 46 variables for each pull, so that's:
+ dg1: ID's 1 - 46  
+ dg2: ID's 47-92  
+ dg3: ID's 93-138  
+ dg4: ID's 139-184  
+ dg5: ID's 185-230  
+ dg6: ID's 231-276  
+ dg7: ID's 277-322  
+ dg8: ID's 323-368  
+ dg9: ID's 369-414  
+ dg10: ID's 415-460  
+ dg11: ID's 461-506  
+ dg12: ID's 507-552  
+ dg13: ID's 553-598  
+ dg14: ID's 599-644  
+ dg15: ID's 645-690  
+ dg16: ID's 691-736  
+ dg17: ID's 737-782  
+ dg18: ID's 783-828  
**This data guide stops at ID 738 as of 5/2/2022.**

In [7]:
dataguide = pd.read_csv('../DATA GUIDE ACS.csv', dtype = str)
dataguide['ID'] = dataguide['ID'].astype(int)

In [63]:
dg1 = dataguide[dataguide['ID'].between(1, 46)]
dg2 = dataguide[dataguide['ID'].between(47, 92)]
dg3 = dataguide[dataguide['ID'].between(93, 138)]
dg4 = dataguide[dataguide['ID'].between(139, 184)]
dg5 = dataguide[dataguide['ID'].between(185, 230)]
dg6 = dataguide[dataguide['ID'].between(231, 276)]
dg7 = dataguide[dataguide['ID'].between(277, 322)]
dg8 = dataguide[dataguide['ID'].between(323, 368)]
dg9 = dataguide[dataguide['ID'].between(369, 414)]
dg10 = dataguide[dataguide['ID'].between(415, 460)]
dg11 = dataguide[dataguide['ID'].between(461, 506)]
dg12 = dataguide[dataguide['ID'].between(507, 552)]
dg13 = dataguide[dataguide['ID'].between(553, 598)]
dg14 = dataguide[dataguide['ID'].between(599, 644)]
dg15 = dataguide[dataguide['ID'].between(645, 690)]
dg16 = dataguide[dataguide['ID'].between(691, 736)]
dg17 = dataguide[dataguide['ID'].between(737, 782)]
dg18 = dataguide[dataguide['ID'].between(783, 828)]

## Loop through all dataguides and geographies in one API call

In [64]:
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'CountyFIPS'

In [65]:
#dflist = [dg1,dg2,dg3,dg4,dg5,dg6,dg7,dg8,dg9,dg10,dg11,dg12,dg13,dg14,dg15,dg16,dg17,dg18]

In [66]:
dflist = [dg1,dg2, dg3]

In [55]:
# TESTING
data = []
for index, element in enumerate(dflist):
    var_list = list(dflist[index]['ACS Variable'])
    #add stuff to variables list
    var_list = deque(var_list)
    var_list.appendleft(head2)
    var_list.appendleft(head1)
    var_list = list(var_list)
    #make columns list
    col_list = list(dflist[index]['Column Name'])
    #add stuff to columns list
    col_list.append(tail_cols1)
    col_list.append(tail_cols2)
    col_list = deque(col_list)
    col_list.appendleft(head2)
    col_list.appendleft(head1)
    col_list = list(col_list)
    #API call
    data_appended = []
    for i in GNRC:
        url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
        predicates= {}
        get_vars= var_list
        predicates["get"]= ",". join(get_vars)
        predicates["for"]= "county:{}".format(i)
        predicates["in"]= "state:47"
        data= requests.get(url_str, params= predicates)
        col_names = col_list
        data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
        data['Region'] = 'GNRC'
        data_appended.append(data)
    data_appended = pd.concat(data_appended)
    dflist[index] = data_appended.reset_index(drop = True)
data = dflist[index-1].merge(dflist[index], how = "outer", on = 'GEO_ID')
print('Your API call is complete.')

TypeError: first argument must be an iterable of pandas objects, you passed an object of type "DataFrame"

In [50]:
data.head()

,NAME_x,GEO_ID,age_f_70to74,age_f_75to79,age_f_80to84,age_f_85+,raceeth_white_alone,raceeth_blackafricanamerican_alone,raceeth_americanindianalaskanative_alone,raceeth_asian_alone,raceeth_nativehawaiianotherpacificislander_alone,raceeth_someotherrace_alone,raceeth_twoormoreraces_alone,raceeth_whitealone_nothispanicorlatino,raceeth_hispanicorlatino,hhsize_avg,healthcoverage_total_healthcare_series,healthcoverage_total_mallhealthcare,healthcoverage_mu6_all,healthcoverage_mu6_w,healthcoverage_mu6_wo,healthcoverage_m6to18_all,healthcoverage_m6to18_w,healthcoverage_m6to18_wo,healthcoverage_m19to25_all,healthcoverage_m19to25_w,healthcoverage_m19to25_wo,healthcoverage_m26to34_all,healthcoverage_m26to34_w,healthcoverage_m26to34_wo,healthcoverage_m35to44_all,healthcoverage_m35to44_w,healthcoverage_m35to44_wo,healthcoverage_m45to54_all,healthcoverage_m45to54_w,healthcoverage_m45to54_wo,healthcoverage_m55to64_all,healthcoverage_m55to64_w,healthcoverage_m55to64_wo,healthcoverage_m65to74_all,healthcoverage_m65to74_w,healthcoverage_m65to74_wo,healthcoverage_m75+_all,healthcoverage_m75+_w,healthcoverage_m75+_wo,healthcoverage_total_fallhealthcare,healthcoverage_fu6_all,healthcoverage_fu6_w,StateFIPS_x,CountyFIPS_x,Region_x,NAME_y,healthcoverage_fu6_wo,healthcoverage_f6to18_all,healthcoverage_f6to18_w,healthcoverage_f6to18_wo,healthcoverage_f19to25_all,healthcoverage_f19to25_w,healthcoverage_f19to25_wo,healthcoverage_f26to34_all,healthcoverage_f26to34_w,healthcoverage_f26to34_wo,healthcoverage_f35to44_all,healthcoverage_f35to44_w,healthcoverage_f35to44_wo,healthcoverage_f45to54_all,healthcoverage_f45to54_w,healthcoverage_f45to54_wo,healthcoverage_f55to64_all,healthcoverage_f55to64_w,healthcoverage_f55to64_wo,healthcoverage_f65to74_all,healthcoverage_f65to74_w,healthcoverage_f65to74_wo,healthcoverage_f75+_all,healthcoverage_f75+_w,healthcoverage_f75+_wo,healthcoverage_total_privatehealthcare_series,healthcoverage_total_mprivate,healthcoverage_mu6_privateall,healthcoverage_mu6_wprivate,healthcoverage_mu6_woprivate,healthcoverage_m6to18_privateall,healthcoverage_m6to18_wprivate,healthcoverage_m6to18_woprivate,healthcoverage_m19to25_privateall,healthcoverage_m19to25_wprivate,healthcoverage_m19to25_woprivate,healthcoverage_m26to34_privateall,healthcoverage_m26to34_wprivate,healthcoverage_m26to34_woprivate,healthcoverage_m35to44_privateall,healthcoverage_m35to44_wprivate,healthcoverage_m35to44_woprivate,healthcoverage_m45to54_privateall,healthcoverage_m45to54_wprivate,healthcoverage_m45to54_woprivate,healthcoverage_m55to64_privateall,StateFIPS_y,CountyFIPS_y,Region_y
0,"Stewart County, Tennessee",0500000US47161,345,289,250,120,12685,49,70,71,111,89,478,12397,433,2.57,13357,6433,435,403,32,1116,1030,86,517,487,30,590,388,202,733,635,98,908,668,240,964,867,97,752,752,0,418,418,0,6924,422,393,47,161,GNRC,"Stewart County, Tennessee",29,1162,1117,45,424,424,0,731,685,46,780,747,33,957,847,110,1019,958,61,802,802,0,627,627,0,13357,6433,435,244,191,1116,669,447,517,369,148,590,299,291,733,477,256,908,573,335,964,47,161,GNRC
1,"Montgomery County, Tennessee",0500000US47125,2656,1754,1130,1344,141679,40546,947,4878,663,3826,12453,128270,21063,2.70,191007,90431,10460,9930,530,18790,17927,863,8904,7187,1717,13456,10980,2476,11149,9489,1660,10328,9341,987,8992,8246,746,5465,5421,44,2887,2854,33,100576,9537,9360,47,125,GNRC,"Montgomery County, Tennessee",177,18427,17670,757,11188,10075,1113,16025,14489,1536,13477,12446,1031,11377,10684,693,10354,9609,745,6264,6221,43,3927,3927,0,191007,90431,10460,7369,3091,18790,13221,5569,8904,6576,2328,13456,9363,4093,11149,8273,2876,10328,8110,2218,8992,47,125,GNRC
2,"Houston County, Tennessee",0500000US47083,240,192,153,78,7644,391,22,13,20,0,111,7521,124,2.73,7982,3985,360,339,21,777,777,0,208,192,16,435,357,78,407,354,53,501,464,37,578,474,104,441,441,0,278,278,0,3997,247,237,47,083,GNRC,"Houston County, Tennessee",10,557,532,25,332,307,25,408,302,106,453,397,56,559,506,53,590,505,85,489,489,0,362,362,0,7982,3985,360,174,

In [38]:
data_appended.head()

,NAME,GEO_ID,healthcoverage_fu6_wo,healthcoverage_f6to18_all,healthcoverage_f6to18_w,healthcoverage_f6to18_wo,healthcoverage_f19to25_all,healthcoverage_f19to25_w,healthcoverage_f19to25_wo,healthcoverage_f26to34_all,healthcoverage_f26to34_w,healthcoverage_f26to34_wo,healthcoverage_f35to44_all,healthcoverage_f35to44_w,healthcoverage_f35to44_wo,healthcoverage_f45to54_all,healthcoverage_f45to54_w,healthcoverage_f45to54_wo,healthcoverage_f55to64_all,healthcoverage_f55to64_w,healthcoverage_f55to64_wo,healthcoverage_f65to74_all,healthcoverage_f65to74_w,healthcoverage_f65to74_wo,healthcoverage_f75+_all,healthcoverage_f75+_w,healthcoverage_f75+_wo,healthcoverage_total_privatehealthcare_series,healthcoverage_total_mprivate,healthcoverage_mu6_privateall,healthcoverage_mu6_wprivate,healthcoverage_mu6_woprivate,healthcoverage_m6to18_privateall,healthcoverage_m6to18_wprivate,healthcoverage_m6to18_woprivate,healthcoverage_m19to25_privateall,healthcoverage_m19to25_wprivate,healthcoverage_m19to25_woprivate,healthcoverage_m26to34_privateall,healthcoverage_m26to34_wprivate,healthcoverage_m26to34_woprivate,healthcoverage_m35to44_privateall,healthcoverage_m35to44_wprivate,healthcoverage_m35to44_woprivate,healthcoverage_m45to54_privateall,healthcoverage_m45to54_wprivate,healthcoverage_m45to54_woprivate,healthcoverage_m55to64_privateall,StateFIPS,CountyFIPS,Region
0,"Stewart County, Tennessee",0500000US47161,29,1162,1117,45,424,424,0,731,685,46,780,747,33,957,847,110,1019,958,61,802,802,0,627,627,0,13357,6433,435,244,191,1116,669,447,517,369,148,590,299,291,733,477,256,908,573,335,964,47,161,GNRC
0,"Montgomery County, Tennessee",0500000US47125,177,18427,17670,757,11188,10075,1113,16025,14489,1536,13477,12446,1031,11377,10684,693,10354,9609,745,6264,6221,43,3927,3927,0,191007,90431,10460,7369,3091,18790,13221,5569,8904,6576,2328,13456,9363,4093,11149,8273,2876,10328,8110,2218,8992,47,125,GNRC
0,"Houston County, Tennessee",0500000US47083,10,557,532,25,332,307,25,408,302,106,453,397,56,559,506,53,590,505,85,489,489,0,362,362,0,7982,3985,360,174,186,777,467,310,208,138,70,435,305,130,407,232,175,501,402,99,578,47,083,GNRC
0,"Humphreys County, Tennessee",0500000US47085,1,1603,1603,0,605,503,102,922,883,39,1140,918,222,1281,1129,152,1398,1077,321,1045,1045,0,859,859,0,18285,8942,675,326,349,1472,1059,413,674,483,191,1005,636,369,1075,701,374,1155,1023,132,1267,47,085,GNRC
0,"Dickson County, Tennessee",0500000US47043,105,4230,4112,118,2093,1737,356,3187,2711,476,3382,2949,433,3626,3105,521,3909,3432,477,2771,2771,0,1909,1909,0,52932,25999,1951,1022,929,4911,3102,1809,2010,1494,516,2984,2163,821,3256,2230,1026,3616,2916,700,3473,47,043,GNRC


In [248]:
data.head()

,NAME_x,GEO_ID,pop,agebysex_total_series,age_total_male,age_m_u5,age_m_5to9,age_m_10to14,age_m_15to17,age_m_18to19,age_m_20,age_m_21,age_m_22to24,age_m_25to29,age_m_30to34,age_m_35to39,age_m_40to44,age_m_45to49,age_m_50to54,age_m_55to59,age_m_60to61,age_m_62to64,age_m_65to66,age_m_67to69,age_m_70to74,age_m_75to79,age_m_80to84,age_m_85+,age_total_female,age_f_u5,age_f_5to9,age_f_10to14,age_f_15to17,age_f_18to19,age_f_20,age_f_21,age_f_22to24,age_f_25to29,age_f_30to34,age_f_35to39,age_f_40to44,age_f_45to49,age_f_50to54,age_f_55to59,age_f_60to61,age_f_62to64,age_f_65to66,age_f_67to69,StateFIPS_x,CountyFIPS_x,Region_x,NAME_y,age_f_70to74,age_f_75to79,age_f_80to84,age_f_85+,raceeth_white_alone,raceeth_blackafricanamerican_alone,raceeth_americanindianalaskanative_alone,raceeth_asian_alone,raceeth_nativehawaiianotherpacificislander_alone,raceeth_someotherrace_alone,raceeth_twoormoreraces_alone,raceeth_whitealone_nothispanicorlatino,raceeth_hispanicorlatino,hhsize_avg,healthcoverage_total_healthcare_series,healthcoverage_total_mallhealthcare,healthcoverage_mu6_all,healthcoverage_mu6_w,healthcoverage_mu6_wo,healthcoverage_m6to18_all,healthcoverage_m6to18_w,healthcoverage_m6to18_wo,healthcoverage_m19to25_all,healthcoverage_m19to25_w,healthcoverage_m19to25_wo,healthcoverage_m26to34_all,healthcoverage_m26to34_w,healthcoverage_m26to34_wo,healthcoverage_m35to44_all,healthcoverage_m35to44_w,healthcoverage_m35to44_wo,healthcoverage_m45to54_all,healthcoverage_m45to54_w,healthcoverage_m45to54_wo,healthcoverage_m55to64_all,healthcoverage_m55to64_w,healthcoverage_m55to64_wo,healthcoverage_m65to74_all,healthcoverage_m65to74_w,healthcoverage_m65to74_wo,healthcoverage_m75+_all,healthcoverage_m75+_w,healthcoverage_m75+_wo,healthcoverage_total_fallhealthcare,healthcoverage_fu6_all,healthcoverage_fu6_w,StateFIPS_y,CountyFIPS_y,Region_y
0,"Stewart County, Tennessee",0500000US47161,13553,13553,6572,353,476,364,263,121,50,127,177,438,339,303,456,437,494,459,283,227,209,272,291,191,113,129,6981,422,305,419,278,167,115,126,101,418,401,317,463,420,537,470,232,317,222,247,47,161,GNRC,"Stewart County, Tennessee",345,289,250,120,12685,49,70,71,111,89,478,12397,433,2.57,13357,6433,435,403,32,1116,1030,86,517,487,30,590,388,202,733,635,98,908,668,240,964,867,97,752,752,0,418,418,0,6924,422,393,47,161,GNRC
1,"Montgomery County, Tennessee",0500000US47125,204992,204992,102205,8916,7959,7229,4053,2544,1642,1694,5927,10985,9174,7666,6094,5506,5259,4623,2116,2378,1684,1912,1930,1315,866,733,102787,8202,7512,7228,3840,2668,2397,1482,4333,10075,8753,7521,6207,5940,5600,5312,2537,2555,2069,1672,47,125,GNRC,"Montgomery County, Tennessee",2656,1754,1130,1344,141679,40546,947,4878,663,3826,12453,128270,21063,2.70,191007,90431,10460,9930,530,18790,17927,863,8904,7187,1717,13456,10980,2476,11149,9489,1660,10328,9341,987,8992,8246,746,5465,5421,44,2887,2854,33,100576,9537,9360,47,125,GNRC
2,"Houston County, Tennessee",0500000US47083,8201,8201,4119,216,347,234,213,204,45,19,53,244,219,186,267,260,256,311,175,106,73,169,237,164,54,67,4082,192,295,190,94,82,51,68,91,262,219,164,289,292,275,295,113,182,89,176,47,083,GNRC,"Houston County, Tennessee",240,192,153,78,7644,391,22,13,20,0,111,7521,124,2.73,7982,3985,360,339,21,777,777,0,208,192,16,435,357,78,407,354,53,501,464,37,578,474,104,441,441,0,278,278,0,3997,247,237,47,083,GNRC
3,"Humphreys County, Tennessee",0500000US47085,18528,18528,9113,538,565,663,353,193,74,130,373,595,470,573,502,564,612,685,271,333,198,284,502,398,177,60,9415,424,494,659,308,295,77,69,283,527,486,726,414,595,686,812,179,407,417,275,47,085,GNRC,"Humphreys County, Tennessee",394,509,211,168,17119,385,148,137,9,186,544,16915,469,2.66,18285,8942,675,666,9,1472,1462,10,674,553,121,1005,749,256,1075,941,134,1155,1097,58,1267,1207,60,984,984,0,635,635,0,9343,490,489,47,085,GNRC
4,"Dickson County, Tennessee",0500000US47043,53289,53289,26246,1733,1662,1850,1196,594,502,204,702,1750,1773,2077,1255,1752,1889,1571,801,1120,615,768,1056,667,460,249,27043,1471,1793,157

In [242]:
# # THIS WORKS FOR TWO DATA GUIDES IN THE LIST~!!!!!!!!!!!!!!
# for index, element in enumerate(dflist):
#     var_list = list(dflist[index]['ACS Variable'])
#     #add stuff to variables list
#     var_list = deque(var_list)
#     var_list.appendleft(head2)
#     var_list.appendleft(head1)
#     var_list = list(var_list)
#     #make columns list
#     col_list = list(dflist[index]['Column Name'])
#     #add stuff to columns list
#     col_list.append(tail_cols1)
#     col_list.append(tail_cols2)
#     col_list = deque(col_list)
#     col_list.appendleft(head2)
#     col_list.appendleft(head1)
#     col_list = list(col_list)
#     #API call
#     data_appended = []
#     for i in GNRC:
#         url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
#         predicates= {}
#         get_vars= var_list
#         predicates["get"]= ",". join(get_vars)
#         predicates["for"]= "county:{}".format(i)
#         predicates["in"]= "state:47"
#         data= requests.get(url_str, params= predicates)
#         col_names = col_list
#         data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
#         data['Region'] = 'GNRC'
#         data_appended.append(data)
#     data_appended = pd.concat(data_appended)
#     dflist[index] = data_appended.reset_index(drop = True)
# data = dflist[index-1].merge(dflist[index], how = "outer", on = 'GEO_ID')
# print('Your API call is complete.')

In [67]:
# THIS WORKS FOR LAST TWO ELEMENTS IN LIST
for index, element in enumerate(dflist):
    var_list = list(dflist[index]['ACS Variable'])
    #add stuff to variables list
    var_list = deque(var_list)
    var_list.appendleft(head2)
    var_list.appendleft(head1)
    var_list = list(var_list)
    #make columns list
    col_list = list(dflist[index]['Column Name'])
    #add stuff to columns list
    col_list.append(tail_cols1)
    col_list.append(tail_cols2)
    col_list = deque(col_list)
    col_list.appendleft(head2)
    col_list.appendleft(head1)
    col_list = list(col_list)
    #API call
    data_appended = []
    for i in GNRC:
        url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
        predicates= {}
        get_vars= var_list
        predicates["get"]= ",". join(get_vars)
        predicates["for"]= "county:{}".format(i)
        predicates["in"]= "state:47"
        data= requests.get(url_str, params= predicates)
        col_names = col_list
        data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
        data['Region'] = 'GNRC'
        data_appended.append(data)
    data_appended = pd.concat(data_appended)
    dflist[index] = data_appended.reset_index(drop = True)
data = dflist[index-1].merge(dflist[index], how = "outer", on = 'GEO_ID')
data_appended = data
print('Your API call is complete.')

Your API call is complete.


In [61]:
data.head()

,NAME_x,GEO_ID,age_f_70to74,age_f_75to79,age_f_80to84,age_f_85+,raceeth_white_alone,raceeth_blackafricanamerican_alone,raceeth_americanindianalaskanative_alone,raceeth_asian_alone,raceeth_nativehawaiianotherpacificislander_alone,raceeth_someotherrace_alone,raceeth_twoormoreraces_alone,raceeth_whitealone_nothispanicorlatino,raceeth_hispanicorlatino,hhsize_avg,healthcoverage_total_healthcare_series,healthcoverage_total_mallhealthcare,healthcoverage_mu6_all,healthcoverage_mu6_w,healthcoverage_mu6_wo,healthcoverage_m6to18_all,healthcoverage_m6to18_w,healthcoverage_m6to18_wo,healthcoverage_m19to25_all,healthcoverage_m19to25_w,healthcoverage_m19to25_wo,healthcoverage_m26to34_all,healthcoverage_m26to34_w,healthcoverage_m26to34_wo,healthcoverage_m35to44_all,healthcoverage_m35to44_w,healthcoverage_m35to44_wo,healthcoverage_m45to54_all,healthcoverage_m45to54_w,healthcoverage_m45to54_wo,healthcoverage_m55to64_all,healthcoverage_m55to64_w,healthcoverage_m55to64_wo,healthcoverage_m65to74_all,healthcoverage_m65to74_w,healthcoverage_m65to74_wo,healthcoverage_m75+_all,healthcoverage_m75+_w,healthcoverage_m75+_wo,healthcoverage_total_fallhealthcare,healthcoverage_fu6_all,healthcoverage_fu6_w,StateFIPS_x,CountyFIPS_x,Region_x,NAME_y,healthcoverage_fu6_wo,healthcoverage_f6to18_all,healthcoverage_f6to18_w,healthcoverage_f6to18_wo,healthcoverage_f19to25_all,healthcoverage_f19to25_w,healthcoverage_f19to25_wo,healthcoverage_f26to34_all,healthcoverage_f26to34_w,healthcoverage_f26to34_wo,healthcoverage_f35to44_all,healthcoverage_f35to44_w,healthcoverage_f35to44_wo,healthcoverage_f45to54_all,healthcoverage_f45to54_w,healthcoverage_f45to54_wo,healthcoverage_f55to64_all,healthcoverage_f55to64_w,healthcoverage_f55to64_wo,healthcoverage_f65to74_all,healthcoverage_f65to74_w,healthcoverage_f65to74_wo,healthcoverage_f75+_all,healthcoverage_f75+_w,healthcoverage_f75+_wo,healthcoverage_total_privatehealthcare_series,healthcoverage_total_mprivate,healthcoverage_mu6_privateall,healthcoverage_mu6_wprivate,healthcoverage_mu6_woprivate,healthcoverage_m6to18_privateall,healthcoverage_m6to18_wprivate,healthcoverage_m6to18_woprivate,healthcoverage_m19to25_privateall,healthcoverage_m19to25_wprivate,healthcoverage_m19to25_woprivate,healthcoverage_m26to34_privateall,healthcoverage_m26to34_wprivate,healthcoverage_m26to34_woprivate,healthcoverage_m35to44_privateall,healthcoverage_m35to44_wprivate,healthcoverage_m35to44_woprivate,healthcoverage_m45to54_privateall,healthcoverage_m45to54_wprivate,healthcoverage_m45to54_woprivate,healthcoverage_m55to64_privateall,StateFIPS_y,CountyFIPS_y,Region_y
0,"Stewart County, Tennessee",0500000US47161,345,289,250,120,12685,49,70,71,111,89,478,12397,433,2.57,13357,6433,435,403,32,1116,1030,86,517,487,30,590,388,202,733,635,98,908,668,240,964,867,97,752,752,0,418,418,0,6924,422,393,47,161,GNRC,"Stewart County, Tennessee",29,1162,1117,45,424,424,0,731,685,46,780,747,33,957,847,110,1019,958,61,802,802,0,627,627,0,13357,6433,435,244,191,1116,669,447,517,369,148,590,299,291,733,477,256,908,573,335,964,47,161,GNRC
1,"Montgomery County, Tennessee",0500000US47125,2656,1754,1130,1344,141679,40546,947,4878,663,3826,12453,128270,21063,2.70,191007,90431,10460,9930,530,18790,17927,863,8904,7187,1717,13456,10980,2476,11149,9489,1660,10328,9341,987,8992,8246,746,5465,5421,44,2887,2854,33,100576,9537,9360,47,125,GNRC,"Montgomery County, Tennessee",177,18427,17670,757,11188,10075,1113,16025,14489,1536,13477,12446,1031,11377,10684,693,10354,9609,745,6264,6221,43,3927,3927,0,191007,90431,10460,7369,3091,18790,13221,5569,8904,6576,2328,13456,9363,4093,11149,8273,2876,10328,8110,2218,8992,47,125,GNRC
2,"Houston County, Tennessee",0500000US47083,240,192,153,78,7644,391,22,13,20,0,111,7521,124,2.73,7982,3985,360,339,21,777,777,0,208,192,16,435,357,78,407,354,53,501,464,37,578,474,104,441,441,0,278,278,0,3997,247,237,47,083,GNRC,"Houston County, Tennessee",10,557,532,25,332,307,25,408,302,106,453,397,56,559,506,53,590,505,85,489,489,0,362,362,0,7982,3985,360,174,

In [68]:
data_appended.head()

,NAME_x,GEO_ID,age_f_70to74,age_f_75to79,age_f_80to84,age_f_85+,raceeth_white_alone,raceeth_blackafricanamerican_alone,raceeth_americanindianalaskanative_alone,raceeth_asian_alone,raceeth_nativehawaiianotherpacificislander_alone,raceeth_someotherrace_alone,raceeth_twoormoreraces_alone,raceeth_whitealone_nothispanicorlatino,raceeth_hispanicorlatino,hhsize_avg,healthcoverage_total_healthcare_series,healthcoverage_total_mallhealthcare,healthcoverage_mu6_all,healthcoverage_mu6_w,healthcoverage_mu6_wo,healthcoverage_m6to18_all,healthcoverage_m6to18_w,healthcoverage_m6to18_wo,healthcoverage_m19to25_all,healthcoverage_m19to25_w,healthcoverage_m19to25_wo,healthcoverage_m26to34_all,healthcoverage_m26to34_w,healthcoverage_m26to34_wo,healthcoverage_m35to44_all,healthcoverage_m35to44_w,healthcoverage_m35to44_wo,healthcoverage_m45to54_all,healthcoverage_m45to54_w,healthcoverage_m45to54_wo,healthcoverage_m55to64_all,healthcoverage_m55to64_w,healthcoverage_m55to64_wo,healthcoverage_m65to74_all,healthcoverage_m65to74_w,healthcoverage_m65to74_wo,healthcoverage_m75+_all,healthcoverage_m75+_w,healthcoverage_m75+_wo,healthcoverage_total_fallhealthcare,healthcoverage_fu6_all,healthcoverage_fu6_w,StateFIPS_x,CountyFIPS_x,Region_x,NAME_y,healthcoverage_fu6_wo,healthcoverage_f6to18_all,healthcoverage_f6to18_w,healthcoverage_f6to18_wo,healthcoverage_f19to25_all,healthcoverage_f19to25_w,healthcoverage_f19to25_wo,healthcoverage_f26to34_all,healthcoverage_f26to34_w,healthcoverage_f26to34_wo,healthcoverage_f35to44_all,healthcoverage_f35to44_w,healthcoverage_f35to44_wo,healthcoverage_f45to54_all,healthcoverage_f45to54_w,healthcoverage_f45to54_wo,healthcoverage_f55to64_all,healthcoverage_f55to64_w,healthcoverage_f55to64_wo,healthcoverage_f65to74_all,healthcoverage_f65to74_w,healthcoverage_f65to74_wo,healthcoverage_f75+_all,healthcoverage_f75+_w,healthcoverage_f75+_wo,healthcoverage_total_privatehealthcare_series,healthcoverage_total_mprivate,healthcoverage_mu6_privateall,healthcoverage_mu6_wprivate,healthcoverage_mu6_woprivate,healthcoverage_m6to18_privateall,healthcoverage_m6to18_wprivate,healthcoverage_m6to18_woprivate,healthcoverage_m19to25_privateall,healthcoverage_m19to25_wprivate,healthcoverage_m19to25_woprivate,healthcoverage_m26to34_privateall,healthcoverage_m26to34_wprivate,healthcoverage_m26to34_woprivate,healthcoverage_m35to44_privateall,healthcoverage_m35to44_wprivate,healthcoverage_m35to44_woprivate,healthcoverage_m45to54_privateall,healthcoverage_m45to54_wprivate,healthcoverage_m45to54_woprivate,healthcoverage_m55to64_privateall,StateFIPS_y,CountyFIPS_y,Region_y
0,"Stewart County, Tennessee",0500000US47161,345,289,250,120,12685,49,70,71,111,89,478,12397,433,2.57,13357,6433,435,403,32,1116,1030,86,517,487,30,590,388,202,733,635,98,908,668,240,964,867,97,752,752,0,418,418,0,6924,422,393,47,161,GNRC,"Stewart County, Tennessee",29,1162,1117,45,424,424,0,731,685,46,780,747,33,957,847,110,1019,958,61,802,802,0,627,627,0,13357,6433,435,244,191,1116,669,447,517,369,148,590,299,291,733,477,256,908,573,335,964,47,161,GNRC
1,"Montgomery County, Tennessee",0500000US47125,2656,1754,1130,1344,141679,40546,947,4878,663,3826,12453,128270,21063,2.70,191007,90431,10460,9930,530,18790,17927,863,8904,7187,1717,13456,10980,2476,11149,9489,1660,10328,9341,987,8992,8246,746,5465,5421,44,2887,2854,33,100576,9537,9360,47,125,GNRC,"Montgomery County, Tennessee",177,18427,17670,757,11188,10075,1113,16025,14489,1536,13477,12446,1031,11377,10684,693,10354,9609,745,6264,6221,43,3927,3927,0,191007,90431,10460,7369,3091,18790,13221,5569,8904,6576,2328,13456,9363,4093,11149,8273,2876,10328,8110,2218,8992,47,125,GNRC
2,"Houston County, Tennessee",0500000US47083,240,192,153,78,7644,391,22,13,20,0,111,7521,124,2.73,7982,3985,360,339,21,777,777,0,208,192,16,435,357,78,407,354,53,501,464,37,578,474,104,441,441,0,278,278,0,3997,247,237,47,083,GNRC,"Houston County, Tennessee",10,557,532,25,332,307,25,408,302,106,453,397,56,559,506,53,590,505,85,489,489,0,362,362,0,7982,3985,360,174,

In [ ]:
c3 = c3.drop(columns = [head1, tail_cols1, tail_cols2])
c4 = c4.drop(columns = [head1, tail_cols1, tail_cols2])
c5 = c5.drop(columns = [head1, tail_cols1, tail_cols2])
c6 = c6.drop(columns = [head1, tail_cols1, tail_cols2])
c7 = c7.drop(columns = [head1, tail_cols1, tail_cols2])
c8 = c8.drop(columns = [head1, tail_cols1, tail_cols2])
c9 = c9.drop(columns = [head1, tail_cols1, tail_cols2])
c10 = c10.drop(columns = [head1, tail_cols1, tail_cols2])
c11 = c11.drop(columns = [head1, tail_cols1, tail_cols2])
c12 = c12.drop(columns = [head1, tail_cols1, tail_cols2])
c13 = c13.drop(columns = [head1, tail_cols1, tail_cols2])
c14 = c14.drop(columns = [head1, tail_cols1, tail_cols2])
c15 = c15.drop(columns = [head1, tail_cols1, tail_cols2])
c16 = c16.drop(columns = [head1, tail_cols1, tail_cols2])
c17 = c17.drop(columns = [head1, tail_cols1, tail_cols2])
c18 = c18.drop(columns = [head1, tail_cols1, tail_cols2])

In [ ]:
dfs = [c2, c3, c4, c5, c6, c7, c8, c9, c10, c11, c12, c13, c14, c15, c16, c17, finalcall]

In [ ]:
df_merged = reduce(lambda  left,right: pd.merge(left,right,on=['GEO_ID'],
                                            how='outer'), dfs)